In [1]:
import json

%load_ext autoreload
%autoreload 2

from utgen.raggraph.walker import build_graph_from_directory
from utgen.raggraph.utils import get_node_context
from utgen.test_generation_crew.crew import TestGenerationCrew
from utgen.validation import validar_test_individual, guardar_i_netejar_tests

In [2]:
g = build_graph_from_directory(code_path="../src", save_graph=False)

In [3]:
# Print nodes for inspection
for n, data in list(g.nodes(data=True))[10:20]:
    print(n, "\n\t", json.dumps(data, indent=2))
    print()

utgen/validation.py::function::guardar_i_netejar_tests 
	 {
  "type": "function",
  "name": "guardar_i_netejar_tests",
  "file": "utgen/validation.py",
  "signature": "def guardar_i_netejar_tests(tests_valids, desti)",
  "docstring": "tests_valids: Llista de tuples [(import, codi), ...]",
  "source": "def guardar_i_netejar_tests(tests_valids, desti=\"../tests/test_generats_llm.py\"):\n    \"\"\"\n    tests_valids: Llista de tuples [(import, codi), ...]\n    \"\"\"\n    if not tests_valids:\n        print(\"No hi ha tests v\u00e0lids per guardar.\")\n        return\n\n    os.makedirs(os.path.dirname(desti), exist_ok=True)\n\n    # 1. SEPAREM EN DOS BLOCS\n    bloc_imports = []\n    bloc_tests = []\n    \n    for imp, code in tests_valids:\n        bloc_imports.append(imp.strip())\n        bloc_tests.append(code.strip())\n    \n    # 2. CONCATENEM: primer TOTS els imports, despr\u00e9s els tests\n    contingut_final = \"\\n\".join(bloc_imports) + \"\\n\\n\" + \"\\n\\n\".join(bloc_tests)\

In [4]:
tests_results: dict[str, dict[str, dict[str, str]]] = {}

# TODO: afegir guardrails que falten
test_generator = TestGenerationCrew(guardrail_max_retries=3, verbose=False)

# Per a cada node del graf, obtenim el seu context i creem el test
for node_id, data in list(g.nodes(data=True))[10:20]:
    if data["type"] in ["function", "method"]:
        print(f"Generating tests for node: {node_id}")
        # Get context
        context = get_node_context(g=g, node_id=node_id)

        # Generate tests
        inputs = {"graph_context": context}
        response = test_generator.crew().kickoff(inputs=inputs)

        # Convert string to dictionary
        response_dict = json.loads(response.raw)

        # Store results
        k = data["file"].replace("/", "___").replace(".py", f"___{data['name']}")
        tests_results[k] = response_dict['tests']

Generating tests for node: utgen/validation.py::function::guardar_i_netejar_tests
2026-03-16 00:13:13.642 | DEBUG    | utgen.test_generation_crew.guardrails:validate_tests_schema:44 - Guardrail input:
```json
{
  "tests": {
    "test_guardar_i_netejar_tests_no_tests": {
      "name": "test_guardar_i_netejar_tests_no_tests",
      "imports": "import pytest\nfrom unittest.mock import patch",
      "code": "@patch('os.makedirs')\n@patch('builtins.print')\ndef test_guardar_i_netejar_tests_no_tests(mock_print, mock_makedirs):\n    from utgen.validation import guardar_i_netejar_tests\n    \n    guardar_i_netejar_tests([])\n    \n    mock_print.assert_called_once_with(\"No hi ha tests vàlids per guardar.\")\n    mock_makedirs.assert_not_called()"
    },
    "test_guardar_i_netejar_tests_basic": {
      "name": "test_guardar_i_netejar_tests_basic",
      "imports": "import pytest\nfrom unittest.mock import patch",
      "code": "@patch('os.makedirs')\n@patch('builtins.print')\n@patch('subproce

In [5]:
for k1, v1 in tests_results.items():
    # Validem el test generat
    print(f"\nValidant tests per `{k1}`...")
    base_import = 'from ' + '.'.join(k1.split("___")[:-1]) + ' import ' + k1.split("___")[-1].split(".")[0]
    print(f"Base import afegit: {base_import}")
    filename = f"test_{k1}.py"
    tests_que_han_passat = []

    for k2, v2 in v1.items():
        name, imp, code = k2, v2['imports'], v2['code']
        # Afegim el import base per si no està present
        imp = imp + f"\n{base_import}"

        if validar_test_individual(imp, code):
            print(f"✅ Test `{name}` acceptat")
            tests_que_han_passat.append((imp, code))
        else:
            print(f"❌ Test `{name}` rebutjat")
        print(imp)
        print(code)

    # Guardem i deixem el fitxer perfecte
    guardar_i_netejar_tests(tests_que_han_passat, desti=f"../tests/{filename}")

# TODO: script final que reordene la carpeta tests: un test.py per cada .py del src
# TODO: run pytest coverage


Validant tests per `utgen___validation___guardar_i_netejar_tests`...
Base import afegit: from utgen.validation import guardar_i_netejar_tests
❌ Test `test_guardar_i_netejar_tests_no_tests` rebutjat
import pytest
from unittest.mock import patch
from utgen.validation import guardar_i_netejar_tests
@patch('os.makedirs')
@patch('builtins.print')
def test_guardar_i_netejar_tests_no_tests(mock_print, mock_makedirs):
    from utgen.validation import guardar_i_netejar_tests
    
    guardar_i_netejar_tests([])
    
    mock_print.assert_called_once_with("No hi ha tests vàlids per guardar.")
    mock_makedirs.assert_not_called()
❌ Test `test_guardar_i_netejar_tests_basic` rebutjat
import pytest
from unittest.mock import patch
from utgen.validation import guardar_i_netejar_tests
@patch('os.makedirs')
@patch('builtins.print')
@patch('subprocess.run')
def test_guardar_i_netejar_tests_basic(mock_subprocess, mock_print, mock_makedirs):
    from utgen.validation import guardar_i_netejar_tests
    
 